# Exercise 1: image → JSON → DataFrame

## Tasks

1. **One image:** send a single picture to the OpenAI API and read its description.
2. **Structure:** ask for a fixed JSON format with a few fields.
3. **More images:** repeat for a small folder and collect the answers in a table.
4. **Evaluate:** compare the table with hand labels, field by field.

**This is the solution.** Try `image_extraction.ipynb` first.

## Setup

Select the **Python 3.11** kernel. The OpenAI API key comes from your `OPENAI_API_KEY` Codespaces secret.

This notebook uses the **OpenAI API** (model `gpt-6-luna`). Other providers, e.g. Google Gemini or Anthropic Claude, have their own APIs with different code. The API calls are in `helper.py` – open it to see how they work.

In [ ]:
# Standard Python libraries.
import sys
from typing import Literal

# Libraries installed from requirements.txt.
import jiwer
import pandas as pd
from pydantic import BaseModel
from sklearn.metrics import accuracy_score, f1_score
from sklearn.preprocessing import MultiLabelBinarizer

# helper.py is in the main folder, one level up.
sys.path.append("..")

In [ ]:
# Functions from helper.py.
from helper import DATA, ask, describe, last_usage, show_images

## 1. Send one image to the OpenAI API and read its description

Pick one image. Other images in `data/images/`: `street.jpg`, `sheep.jpg`, `parking.jpg`, `lake_night.jpg`, `lake_swan.jpg`.

In [ ]:
# The image to describe. Change the filename to try another one.
image_path = DATA / "images" / "lake.jpg"
show_images([image_path])

In [ ]:
# Your question about the image as text.
question = "Describe this image in two sentences."

# Send the question and the image to the OpenAI API.
answer = describe(question, image_path)
print(answer)

The model reads the image as tokens, so every image costs money. Estimate before you run: tokens per image × number of images × price. Section 2 shows how to measure the exact number of tokens.

In [ ]:
# Example: a folder with 1,000 images.
images = 1000
tokens_per_image = 1500  # assumed
price = 0.10  # US$ per 1 million input tokens

# Estimated cost for the whole folder.
cost = images * tokens_per_image * price / 1_000_000
print("Estimated cost: US$", cost)

## 2. Ask for a fixed JSON format

A free text is hard to compare. Now the answer must contain exactly these fields, written as a Python class.

- `text_in_image` is `None` when no text is visible.
- `inferred` holds what the model only **concludes** but cannot see, e.g. the time of day. All other fields hold what is **visible**.

In [ ]:
# The answer format: one line per field.
class ImageRecord(BaseModel):
    scene: Literal["lake", "parking lot", "alpine pasture", "city street"]
    objects: list[str]
    people_count: int
    text_in_image: str | None
    inferred: list[str]

In [ ]:
# The instruction for the model.
PROMPT = """
Describe this image.
Pick the scene from the list.
Name up to five prominent objects as lowercase singular nouns.
Count the visible people.
Copy the most prominent text. Use null if there is no text.
Use [unreadable] if the text cannot be read.
Under inferred, list up to three things you conclude but cannot see,
for example the time of day or the season.
Write unknown if the image cannot tell.
"""

In [ ]:
# Same image, now with a fixed answer format.
answer = ask(PROMPT, ImageRecord, [image_path])
answer

**Exact cost:** replace the assumed 1,500 tokens with real numbers.

1. **Tokens:** every API answer reports how many tokens it used: `response.usage.input_tokens` (your prompt and the image) and `response.usage.output_tokens` (the answer). `helper.py` writes both into `api_log.jsonl` in this folder – open the file to see them for every call. `last_usage()` reads them for the last call.
2. **Price:** look it up for exactly the model you use (here `gpt-6-luna`, see `MODEL` in `helper.py`) on its model page, [GPT-6 Luna model and pricing](https://developers.openai.com/api/docs/models/gpt-6-luna), or in the [OpenAI API pricing](https://developers.openai.com/api/docs/pricing) list. Prices are given in US$ per 1 million tokens, separately for input and output. Prices change – check them before a large run.

In [ ]:
# Exact tokens of the last call: the structured answer above.
usage = last_usage()
print(usage)

# Prices in US$ per 1 million tokens for gpt-6-luna, from
# https://developers.openai.com/api/docs/models/gpt-6-luna
price_in = 0.10
price_out = 0.50

# Exact cost of one image, then of a folder with 1,000 images.
cost_one = (usage["input_tokens"] * price_in + usage["output_tokens"] * price_out) / 1_000_000
print(f"Cost per image: US$ {cost_one:.6f}")
print(f"Cost for 1,000 images: US$ {cost_one * 1000:.2f}")

## 3. Repeat for three images and collect the answers in a table

In [ ]:
# Three images, one per scene: parking lot, alpine pasture, city street.
# Add more filenames from data/images/ to extend the table.
folder = DATA / "images"
files = ["parking.jpg", "sheep.jpg", "street.jpg"]
paths = [folder / name for name in files]
print("number of paths/images:", len(paths))

In [ ]:
# Ask the model about every image.
rows = []
for path in paths:
    row = ask(PROMPT, ImageRecord, [path])
    row["file"] = path.name  # we add the filename ourselves
    rows.append(row)

# One row per image.
extracted = pd.DataFrame(rows)
extracted

In [ ]:
# Save the table as JSON lines: one record per line.
extracted.to_json("extracted.jsonl", orient="records", lines=True)

## 4. Compare the table with the hand labels, field by field

The average hides the weak field – so each field gets its own score.

In [ ]:
# Hand labels: one row per image.
labels = pd.read_csv(folder / "labels.csv")

# Put model answer (_pred) and label (_ref) side by side.
scores = extracted.merge(labels, on="file", suffixes=("_pred", "_ref"))
scores

**Scene and people count:** the answer is either right or wrong. Accuracy is the share of images where model and label agree.

In [ ]:
# Accuracy for the scene.
scene = accuracy_score(scores["scene_ref"], scores["scene_pred"])
print("Scene accuracy:", round(scene, 2))

# Accuracy for the number of people.
ref = scores["people_count_ref"]
pred = scores["people_count_pred"]
people = accuracy_score(ref, pred)
print("People count accuracy:", round(people, 2))

**Objects** are a list, so "right or wrong" is too strict. The F1 score is high only if the model names the right objects (precision) **and** finds all of them (recall).

In [ ]:
# Object lists from the model and from the labels.
pred = scores["objects_pred"]
ref = scores["objects_ref"].str.split(";")  # "boat;water" -> ["boat", "water"]

# Turn each list into a row of 0s and 1s, one column per object name.
encoder = MultiLabelBinarizer()
encoder.fit(list(pred) + list(ref))
y_pred = encoder.transform(pred)
y_ref = encoder.transform(ref)

# F1 per image, averaged over all images.
f1 = f1_score(y_ref, y_pred, average="samples")
print("Object F1:", round(f1, 2))

**Text** is scored with the character error rate (CER): how many characters must be changed, per character of the label? Lower is better.

In [ ]:
# Keep only the images whose label contains text.
with_text = scores[scores["text_in_image_ref"].notna()]

# Label text and model text. No text from the model counts as empty text.
reference = list(with_text["text_in_image_ref"])
prediction = list(with_text["text_in_image_pred"].fillna(""))
print("Label:", reference)
print("Model:", prediction)

# Character error rate.
cer = jiwer.cer(reference, prediction)
print("CER:", round(cer, 2))

Compare the object lists of the model and the labels:

In [ ]:
# Model and label objects side by side.
scores[["file", "objects_pred", "objects_ref"]]

## Check your understanding

Answer briefly using the outputs above. Try first, then open the hints.

1. Why can a perfectly valid JSON answer still be wrong? Use the object lists in the table above.
2. Which field has the lowest score in your run? What makes this field hard for the model?
3. The label says `tree`, the model answers `pine tree`. How does this change the object F1 – and is the model really wrong?
4. Why do we add the filename in Python instead of asking the model for it?
5. You want to extract 10,000 images. What do you check before you start the run?

**Your answers:**

1. …
2. …
3. …
4. …
5. …

<details>
<summary style="color: #B8860B; font-weight: bold; cursor: pointer;">Hints for self-check</summary>

1. The JSON format only checks the form: all fields are there and have the right type. It does not check whether the content is true. A wrong people count or an invented object is still valid JSON.

2. Use your measured scores. Counting people and reading small text are often weak. The scene is easier, because the model only picks one of four choices.

3. It counts as one extra object and one missing object, so the F1 drops – although the answer is reasonable. A clearer prompt, a fixed list of object names, or cleaning the names before scoring helps.

4. Python knows the filename exactly. The model could misspell it, and then the row no longer matches its label. Do not ask the model for something you already know.

5. Estimate the cost first: tokens per image × number of images × price. Test the prompt on a small labelled sample, score each field, and keep the log of all answers.

</details>

### Jupyter notebook --footer info--

In [ ]:
# Print system and version info for this run.
import os
import platform
import socket
from platform import python_version
from datetime import datetime

print('-----------------------------------')
print(os.name.upper())
print(platform.system(), '|', platform.release())
print('Datetime:', datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print('Python Version:', python_version())
print('-----------------------------------')